# **CardiAg: ECG Preprocessing**

Preliminary version of the pipeline for preprocessing raw ECG signal, developed as part of the Brain-Body Analysis Special Interest Group (BBSIG). For the most recent version, please refer to https://martager.github.io/bbsig/code/preprocessing/ecg_preproc.ipynb

To know how to use this ECG Preprocessing notebook, visit our step-by-step tutorial: https://martager.github.io/bbsig/ecg-preprocessing/

Jupyter notebook created by Marta Gerosa, Niket Agrawal, Michael Gaebler, Manisha Biswas, Anthony Ciston, Antonin Fourcade, Emilia Hildebrandt, Helena Mastek, Agata Patyczek, Aleksandra Piejka, Elias Reinwarth, Lucy Roellecke, Yu Hei Shum, Sam Verschooren, & Tomaso Zanardi

Created on: 15 April 2024

Last update (by M. Gerosa): 22 April 2026

If you use this BBSIG pipeline in a publication, please cite us: Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://martager.github.io/bbsig/

## **Pipeline structure**

The following steps are included:

1. **Data import and conversion**: import the BIDS-compliant `_physio.tsv.gz` and `_physio.json` files containing the raw ECG signal and convert them into appropriate formats for later processing stages. Optional step to rescale ECG signal (e.g., by 0.001 with BrainAmp ExG acquired via LSL) if needed. To rescale, set `ecg_scale` to `True`.
2. **(Optional) ECG flipping and filtering**: if needed, flip the ECG signal using NeuroKit2’s `ecg_invert()` function, and/or clean the ECG signal using NeuroKit2’s `signal_clean()` function (50 Hz powerline filter, and 0.5-30 Hz band-pass 4th order Butterworth filters). To flip the ECG signal, set `ecg_flip` to `True`. To clean the ECG signal, set `ecg_filter` to `True`.
3. **R-peak detection**: custom function to detect R-peaks using either Systole’s `ecg_peaks()` with 'sleepecg' methods (default) or NeuroKit2’s `nk.ecg_peaks()` with 'neurokit' method. Additionally, the user can choose to run automated artifact correction. Uncorrected (and automatically corrected) R-peak indices are saved. 
4. **(Manual) R-peak correction and interactive visualization**: manually identify and correct mis-detected R-peaks and/or label bad segments using Systole’s `Editor`(which saves a `corrected.json file` in the `derivatives` folder). To perform the manual correction, set `manual_correct` to `True`. If `interactive_ecg_plot` is set to `True`, create an interactive visualization of ECG signal with R-peaks and/or instantaneous heart rate using Systole’s `plot_raw()` function. Plots can be either shown inside the notebook if `plot_within_notebook` is set to `True`, otherwise they will be opened in the browser as HTML files.
5. **QRS delineation and T-wave detection**: delineate the onsets, peaks and offsets of main QRS features, including T-wave peaks, using NeuroKit2 `ecg_delineate()` function with 'dwt' method.
6. **Data output**: export the `_ecg-cleaned.tsv.gz` (optional) and `_ecg-preproc.json` files in BIDS-compliant format for each subject in `/derivatives/ecg-preproc/sub-xx/`. Additionally, an `_hr-bpm-{correction_type}.tsv.gz` file (optional) can be saved with interpolated HR (in BPM). To export the interpolated HR file, set `hr_interpol` to `True`.

In [ ]:
############## Import modules ##############

import numpy as np
import pandas as pd
import os, json
import neurokit2 as nk
from systole.detection import ecg_peaks, rr_artefacts
from systole.utils import input_conversion, heart_rate 
from systole.plots import plot_raw
from systole.correction import correct_peaks
from systole.interact import Editor
from IPython.display import display
import matplotlib.pyplot as plt
from bokeh.plotting import figure, show, output_notebook

## **Settings: optional pipeline steps**

This section defines a series of variables that can be set to `True` to include the corresponding pipeline steps:

| <div style="width:150px">Variable Name</div> | Function |
| :------------------------------------------- | :------- |
| **`ecg_scale`** (bool) <br>**`ecg_scale_factor`** (numeric) | **ECG re-scaling** (Sect. 1): scales the ECG signal by a user-defined factor (e.g., 0.001 for BrainAmp ExG acquired via LSL) specified under `ecg_scale_factor`. For reference, ECG is canonically measured in millivolts (mV), with R-peak amplitude on average ≤2.0/2.5 mV. |
| **`ecg_flip`** (bool) | **ECG flipping** (Sect. 2): checks whether an ECG signal is inverted, and if so, corrects for this inversion using NeuroKit2’s `nk.ecg_invert()` function. Defaults to `False`. If you are unsure whether the original ECG signal is inverted or not, we recommended setting it to `True`. |
| **`ecg_filter`** (bool) |  **ECG filtering** (Sect. 2): cleans the ECG signal by applying a 50 Hz powerline filter + 0.5-30 Hz band-pass 4th order Butterworth filter using NeuroKit2’s `nk.signal_filter()` function. | 
| **`manual_correct`** (bool) | **Manual correction of R-peaks** (Sect. 4): activates UI for manual correction of extra peaks, missed peaks, and falsely detected peaks, using Systole's `Editor`. The UI can also be used to annotate bad segments. Saves a JSON file with the corrected peaks and bad segments. |
| **`interactive_ecg_plot`** (bool) | **Interactive plot of ECG signal and R-peaks** (Sect. 5): displays an interactive plot of the raw ECG signal with R-peak locations and heart rate using Systole's `plot_raw()` function. | 
| **`hr_interpol`** (bool) | **Save HR interpolation** (Sect. 7): exports interpolated heart rate (HR) values in BPM using Systole's `utils.heart_rate()`, as a TSV file with same sampling rate as original recording. | 

In [ ]:
############## Settings: optional pipeline steps ##############

# For S1: set whether (optional) ECG scaling is needed
# You should aim to have your ECG signal in millivolts (mV), with R-peak amplitude on average ≤2.0/2.5 mV. 
ecg_scale = True
ecg_scale_factor = 0.001    # adjust as needed

# For S2: set whether (optional) pre-processing step of flipping ECG signal is needed (when inverted)
# Attention: if ECG signal is too long, it could return a MemoryError
ecg_flip = False

# For S2: set whether (optional) pre-processing step of cleaning ECG signal is needed
ecg_filter = True

# For S4: set whether manual correction of R-peaks using Systole's Editor is needed
manual_correct = True

# For S4: set whether interactive ECG signal plot using Systole's plot_raw() is needed
interactive_ecg_plot = True

# For S6: set whether data export of TSV file with interpolated HR values (in bpm) is needed
hr_interpol = True

## **1. Data import and conversion**

This section imports the physiological data and metadata from the `_physio.tsv.gz` and sidecar `_physio.json` file from their BIDS directory and extracts the ECG data into a DataFrame (`ecg_df`) and numpy array (`ecg_arr`). In detail:
- Define the name and directory path for the BIDS-compatible `_physio.tsv.gz` and `_physio.json` files containing physiological data and metadata, respectively.
- Import, decompress and store the TSV file as a DataFrame (`physio_df`).
- Extract metadata about column names and sampling frequency from the JSON file.
- Rename columns for cardiac and respiratory data to 'ECG' and 'RSP'.
- Scale the ECG data (optional, if `ecg_scale` is set to `True`) and extracts it into a DataFrame (`ecg_df`) and numpy array (`ecg_arr`).
- Define the path for storing ECG pre-processed data (`/derivatives/ecg-preproc/`).  

In [ ]:
################## 1. Data import and conversion ##################

############## Define path for ECG data ##############

# Define the participant ID
# If manual R-peak correction (Sect. 4) is selected, run this notebook one participant at a time
participant_ids = [1]  # Adjust as needed

# Specify the data path info (in BIDS format)
wd = r'.\data'              # directory of data storage
exp_name = 'CardiAgIBTask'
datatype_name = 'beh'       # current datatype folder according to BIDS

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)
    
    # Specify the name and directory of the BIDS-compatible physio data file
    tsv_fname = f'{subj_id}_task-{exp_name}_physio.tsv.gz'
    tsv_dir = os.path.join(wd, subj_id, datatype_name, tsv_fname)
    
    # Specify the name and directory of JSON file (for info about column names)
    json_fname = f'{subj_id}_task-{exp_name}_physio.json'
    json_dir = os.path.join(wd, subj_id, datatype_name, json_fname)

    # Extract physio metadata from JSON file
    fjson = open(json_dir)
    physio_metadata = json.load(fjson) # load metadata from physio.json
    physio_col = physio_metadata['Columns'] # column names from tsv.gz file 

    # Open physio data file and save as dataframe
    physio_df = pd.read_csv(tsv_dir, compression='gzip', header=None, 
                            sep='\t', names=physio_col) # decompress and read TSV file
    physio_df.rename(columns={'cardiac': 'ECG', 'respiratory': 'RSP'}, inplace=True) # rename columns

    sfreq = physio_metadata['SamplingFrequency'] # save sampling rate
    
    # Extract array and dataframe for ECG data
    if 'ECG' in physio_df.columns:

        if ecg_scale: 
            ecg_df = physio_df['ECG'].copy() * ecg_scale_factor # scale and save ECG df (one data sample per row)
        else:
            ecg_df = physio_df['ECG'].copy() # save ECG df (one data sample per row)
        
        ecg_arr = ecg_df.values # save array with ECG data samples
        

    ############## Define path for ECG preprocessed data ##############
    
    # Check if the BIDS directory 'derivatives/ecg-preproc/' exists, if not create it
    ecg_preproc_folder = 'ecg-preproc'
    ecg_preproc_dir = os.path.join(wd, 'derivatives', ecg_preproc_folder, subj_id, datatype_name)
    if not os.path.exists(ecg_preproc_dir):
        os.makedirs(ecg_preproc_dir)

## **2. (Optional) ECG flipping & filtering**

2a. If the variable `ecg_flip` is set to `True` in the optional pipeline steps (see settings above - default set to `False`), this section:
 
- Checks whether an ECG signal is inverted, and if so, corrects for this inversion using NeuroKit2 `nk.ecg_invert()` function. 
- Stores the resulting flipped ECG signal `ecg_inverted` as `ecg_arr`

2b. If the variable `ecg_filter` is set to `True` in the optional pipeline steps (see settings above - default set to `True`), this section:

- Applies filtering to the ECG signal using NeuroKit2 `nk.signal_filter()` function. Filters include 50 Hz powerline, plus 0.5 Hz high-pass and 30 Hz low-pass 4th order Butterworth filters.
- Stores the resulting clean ECG signal as `ecg_clean` array.

If not enabled, all the subsequent steps have to be conducted on the original raw ECG signal (`ecg_arr`).

In [ ]:
################## 2a. (Optional) ECG flipping ##################

# Proceed only if (optional) pre-processing step of ECG flipping is required
if ecg_flip:

    # Flip the ECG signal if inverted - set force to "False" checks whether the signal is inverted and, if so, flips it
    # If `force` is set to `True`, inversion of the signal is applied regardless of whether it is detected as inverted.
    ecg_inverted, is_inverted = nk.ecg_invert(ecg_arr, sampling_rate=sfreq, show=True, force=False)
    ecg_arr = ecg_inverted.copy()

################## 2b. (Optional) ECG filtering ##################

# Proceed only if (optional) pre-processing step of ECG filtering is required
if ecg_filter:
    
    # Apply filtering to the ECG signal: 50 Hz powerline filter, 
    # 0.5 Hz 4th Butterworth high-pass filter, 30 Hz 4th Butterworth low-pass filter
    ecg_clean = nk.signal_filter(ecg_arr, sampling_rate=sfreq, lowcut=0.5, highcut=30, 
                                 method='butterworth', order=4, powerline=50, show=False)

## **3. R-peak detection**

This section performs R-peak detection on the provided ECG signal and optionally applies automated R-peak correction, based on the chosen method. In detail:

*   Defines a new custom function `detect_rpeaks()` to allow the selection of the preferred method for R-peak extraction, based on a BBSIG-internal validation: the default is Systole `systole.detection.ecg_peaks(method='sleepecg')`, as it performed the most consistently across various tests. Otherwise, the method NeuroKit2 `nk.ecg_peaks(method='neurokit')` can be selected.
*   If `correct_artifacts` is enabled, an automated artifacts correction of irregularities like missed or extra beats is performed. For `'sleepecg'`, the fuction `systole.correction.correct_peaks()` is used; for `'neurokit'`, the correction is performed by the in-built `nk.ecg_peaks(correct_artifacts=True)` argument.


The function returns a dictionary with corrected and uncorrected R-peak indices, R-peak boolean arrays, and information about the detection method and artifacts corrections (see the `detect_rpeaks()` function documentation for more info).

In [ ]:
################## 3. R-peak detection ##################

# Define a custom function for R-peak detection:
# using Systole ecg_peaks(method='sleepecg') as default, otherwise nk.ecg_peaks(method='neurokit')

def detect_rpeaks(signal, sfreq, method='sleepecg', correct_artifacts=True):
    """ Detect R-peaks in ECG signal using specified method ('sleepecg' from Systole or 'neurokit' NeuroKit2) with (optional) automated R-peak correction methods.

        Parameters:
        -----------
        signal : array-like
            The ECG signal (e.g., 'ecg_clean' if filtered, or 'ecg_arr' if raw)
        sfreq : float
            The sampling frequency of the ECG signal.
        method : {'sleepecg', 'neurokit'}, optional
            The method used for R-peak detection. Default is 'sleepecg'.
        correct_artifacts : bool, optional
            Whether to implemented automated artifacts correction in the R-peaks detection. Default is True.

        Returns:
        --------
        dict:
            - 'rpeaks_idx' (array): indices of the detected R-peaks.
            - 'rpeaks_bool' (array): boolean array indicating the presence of R-peaks, of the same length as the input signal.
            - 'info' (dict): additional info about the detection and correction methods. If 'neurokit', it corresponds to the default info dictionary returned by nk.ecg_peaks().
                - 'method_peaks': method used for R-peak detection
                - 'ECG_R_Peaks' (array): peak indices
                - 'ECG_R_Peaks_Uncorrected' (array): uncorrected R-peak indices (only if correct_artifacts=True)
                - 'ectopic_idx_uncorr', 'long_idx_uncorr', 'short_idx_uncorr', 'extra_idx_uncorr', 'missed_idx_uncorr' (arrays): indices of artifacts types before correction. Only if 'sleepecg' is used.
                - 'info_correction' (dict): number of extra and missed peaks corrected. Only if 'sleepecg' is used.

        Raises:
        -------
        ValueError: 
            If an unsupported method other than 'sleepecg' or 'neurokit' is specified.

        Examples:
        ---------
        >>> # Using default method 'sleepecg' for R-peak detection with artifacts correction
        >>> detect_rpeaks(ecg_signal, sfreq, method='sleepecg')
        {'rpeaks_idx': array([ 567,  1307,  2046, ...]),
        'rpeaks_bool': array([False, False, False, ...]),
        'info': {'method_peaks': 'sleepecg',
                'ECG_R_Peaks_Uncorrected': array([ 567,  1310,  2046, ...]
                'ECG_R_Peaks': array([ 567,  1307,  2046, ...]),
                'ectopic_idx_uncorr': array([ 24,  85, 225, ...]),
                'long_idx_uncorr': array([ 194,  201,  251, ...]),
                'short_idx_uncorr': array([ 16,   84,  207, ...]),
                'extra_idx_uncorr': array([ 92,  171,  226, ...]),
                'missed_idx_uncorr': array([ 209,  1264,  2692, ...]),
                'info_correction': {'extra': 2,
                                    'missed': 27}}}
    """

    if method == 'sleepecg': # use Systole ecg_peaks(method='sleepecg')

        # Detect R-peaks
        signal, rpeaks_bool = ecg_peaks(signal=signal, method=method, sfreq=sfreq, new_sfreq=sfreq)

        # Convert from boolean peaks array to peaks indices
        rpeaks = input_conversion(rpeaks_bool, input_type="peaks", output_type="peaks_idx", sfreq=sfreq)

        # Initialize info dict with used method and R-peak indices
        info = {'method_peaks': method,
                'automated_correction': str(correct_artifacts), 
                'ECG_R_Peaks': rpeaks}

        if correct_artifacts: # check if automated artifacts correction is requested

            # Save uncorrected R-peak indices in info dict
            info['ECG_R_Peaks_Uncorrected'] = rpeaks

        # Detect artifacts in RR series (ectopic, long, short, missed, extra)
        artefacts = rr_artefacts(rpeaks_bool, input_type="peaks")

        # Extract idx of each artifacts type before applying correction
        artefacts_types = ['ectopic', 'long', 'short', 'extra', 'missed']
        for artefact in artefacts_types:
            idx = input_conversion(artefacts[artefact], input_type="peaks", output_type="peaks_idx", sfreq=sfreq)

            info[f'{artefact}_idx_uncorr'] = idx

        # Apply automated artifacts correction using Sytole correct_peaks()
        info_correction = correct_peaks(rpeaks_bool)
        rpeaks_bool = info_correction['clean_peaks']
        rpeaks = input_conversion(info_correction['clean_peaks'], input_type="peaks", output_type="peaks_idx", sfreq=sfreq)

        # Save corrected R-peak indices, extra and missed peaks in info dict
        info['ECG_R_Peaks'] = rpeaks
        info['info_correction'] = {
            'extra': info_correction['extra'],
            'missed': info_correction['missed']
        }

        return {'rpeaks_idx': rpeaks,
                'rpeaks_bool': rpeaks_bool,
                'info': info}


    elif method == 'neurokit': # use NeuroKit2 nk.ecg_peaks(method='neurokit')

        # Detect R-peaks
        rpeaks_bin, info = nk.ecg_peaks(signal, sampling_rate=sfreq, method=method,
                                        correct_artifacts=correct_artifacts)

        # Save R-peak indices (corrected or uncorrected, depending on selected option)
        # and boolean vector in info dict
        rpeaks = info['ECG_R_Peaks']
        rpeaks_bool = rpeaks_bin['ECG_R_Peaks'].astype('bool').values

        # Add field for whether automated artifacts correction was performed
        info['automated_correction'] = str(correct_artifacts)

        return {'rpeaks_idx': rpeaks,
                'rpeaks_bool': rpeaks_bool,
                'info': info}

    else:
        raise ValueError("Wrong method! R-peak detection method should be either 'sleepecg' or 'neurokit'")

In [ ]:
# Perform the R-peak detection with the chosen method
# Specify "ecg_clean" if ecg signal filtering was applied, otherwise "ecg_arr"
rpeaks_dict = detect_rpeaks(signal=ecg_clean, sfreq=sfreq, correct_artifacts=True)

rpeaks_dict       # uncomment to view output

## **4. R-peaks manual correction and interactive visualization**

If enabled via `manual_correct`, this section allows the interactive manual correction of R-peak locations and identification of noisy segments in the ECG signal using Systole `Editor`. Both the raw signal and the instantaneous heart rate are plotted to check for artefacts (e.g., long/short beats, ectopic beats). This interactive plot features a "Correction" mode for deleting peaks or adding them at the local maxima within selected segments, and a "Rejection" mode for marking selected segments as bad. The corrected R-peak locations are saved to a JSON file (`_systole-corrected.json`) for further processing and analysis.

If an interactive visualization of the ECG signal with R-peaks and/or the instantaneous heart rate is needed (`interactive_ecg_plot` set to True), this is provided using Systole `plot_raw()` function.

In [ ]:
############## 4. R-peaks manual correction ##############

############# Interactive plot for manual peaks correction ##############

# Execute this first
if manual_correct:

    # Save JSON file with corrected R-peaks and bad segments indices
    ecg_corr_fname = f'{subj_id}_task-{exp_name}_systole-corrected.json'
    ecg_corr_fpath = os.path.join(ecg_preproc_dir, ecg_corr_fname)

    # Display interactive plot
    %matplotlib ipympl

    # Specify "ecg_clean" if ecg signal filtering was applied, otherwise "ecg_arr"
    editor = Editor(signal=ecg_clean, 
                    corrected_json=ecg_corr_fpath,
                    sfreq=sfreq, 
                    corrected_peaks=rpeaks_dict['rpeaks_bool'],     # comment if reloading
                    signal_type="ECG", figsize=(10, 6))

    display(editor.commands_box)

In [ ]:
############# Interactive plot for manual peaks correction: data saving ##############

# Execute only when manual peak correction is done
if manual_correct:
  editor.save()

In [ ]:
############# Interactive plot with ECG signal, R-peaks and HR ##############

# Interactive plot with ECG signal, rpeaks and HR (not functioning with newer Bokeh versions)
# Works when downgraded to bokeh version 3.3.4, https://docs.bokeh.org/en/latest/docs/releases.html
# pip install bokeh==3.3.4

if interactive_ecg_plot:
  
  if manual_correct:
      
      # Check that Systole's JSON file exists and, if so, load it
      if os.path.exists(ecg_corr_fpath):
          with open(ecg_corr_fpath, 'r') as ecg_corr_json:
              corr_rpeaks = json.load(ecg_corr_json)

      # Extract values nested under "ecg" and "corrected_peaks" and convert to boolean
      corr_rpeaks_idx = np.array(corr_rpeaks['ecg']['corrected_peaks'])
      corr_rpeaks_idx_bool = input_conversion(corr_rpeaks_idx, input_type="peaks_idx", output_type="peaks", sfreq=sfreq)

      # Check the lenght difference between corr_rpeaks_idx_bool and ecg signal
      length_diff = len(ecg_clean) - len(corr_rpeaks_idx_bool)
      if length_diff > 0:
         corr_rpeaks_idx_bool = np.append(corr_rpeaks_idx_bool, [False] * length_diff)

      show(
         plot_raw(ecg_clean, peaks=corr_rpeaks_idx_bool, backend='bokeh', modality="ecg", 
                  show_artefacts=True, show_heart_rate=True, figsize=300)
                  )

  else:  
     show(
        plot_raw(ecg_clean, peaks=rpeaks_dict['rpeaks_bool'], backend='bokeh', modality="ecg",
                 show_artefacts=True, show_heart_rate=True, figsize=300)
                 )

## **5. QRS delineation and T-wave detection**

This section delineates the main features of the QRS complex, including the T-wave peak and T-wave offset. It utilizes the `nk.ecg_delineate()` function from NeuroKit2 to perform the QRS delineation. The default delineation method used is `'dwt'` (Discrete Wavelet Transform).

This returns a binary array (`qrs_bin`) indicating the presence of various QRS complex events and a dictionary of indices for QRS-peaks, QRS-onsets and QRS-offsets (`qrs_idx`). Indices of T-wave peaks (`twave_peaks`) and T-wave offsets (`twave_off`) are separately defined for easier access.

In [ ]:
############## 5. QRS delineation & T-wave detection ##############

# If manual correction was performed, load corrected_json
if manual_correct:

    # Check that JSON file exists and, if so, load it
    if os.path.exists(ecg_corr_fpath):
        with open(ecg_corr_fpath, 'r') as ecg_corr_json:
            corr_rpeaks = json.load(ecg_corr_json)
    
    corr_rpeaks_idx = np.array(corr_rpeaks['ecg']['corrected_peaks'])

    # Delineate the main QRS complex features (incl. T-wave peak)
    # using manually corrected R-peaks
    qrs_bin, qrs_idx = nk.ecg_delineate(ecg_clean, rpeaks=corr_rpeaks_idx,
                                        sampling_rate=sfreq, method='dwt')
    
else:
    # Delineate the main QRS complex features (incl. T-wave peak)
    qrs_bin, qrs_idx = nk.ecg_delineate(ecg_clean, rpeaks=rpeaks_dict['rpeaks_idx'],
                                        sampling_rate=sfreq, method='dwt')

# Extract the indices of T-wave peak and offset locations
twave_peaks = qrs_idx['ECG_T_Peaks']
twave_off = qrs_idx['ECG_T_Offsets']

## **6. Data output**

This section exports the ECG pre-processing output files in BIDS-compatible format for each subject in `derivatives/ecg-preproc/sub-xx/`.

- **6a. (Optional) Export TSV.GZ file with raw and clean ECG data**: a custom function `save_ecg_cleaned()` saves the BIDS-compatible `_ecg-cleaned.tsv.gz` file having two columns, `ecg_raw` (i.e., the original ECG data)and `ecg_cleaned` (i.e., the filtered ECG signal, created by the - optional - filtering in S2, if `ecg_filter` is set to `True`).
- **6b. Export JSON file with main ECG-preprocessing features**: a custom function `save_ecg_preproc()` saves the `_ecg-preproc.json` file with R-peaks indices, QRS complex features (nested) and bad segments indices. 
    - `rpeaks` contains the following data: `ECG_R_Peaks_Uncorr`, for uncorrected R-peaks idx; `ECG_R_Peaks_AutoCorr` for auto-corrected R-peaks idx (optional), using either Systole's `correct_peaks()` or NeuroKit's `nk.ecg_peaks(correct_artifacts=True)` method; and `ECG_R_Peaks_ManualCorr` for manually corrected R-peaks using Systole's Editor (optional, only if `manual_correct` is set to True). 
    - `qrs` contains data for delineated QRS complex features, obtained using NeuroKit2 `nk.ecg_delineate()` function, specifically:
        * P-peaks, P-onsets and P-offsets
        * Q-peaks
        * R-onsets and R-offsets
        * S-peaks
        * T-peaks, T-onsets and T-offsets
    - `rr_s` contains the rr intervals time series (in seconds) created using Systole's `input_conversion(output_type='rr_s')`, based on
    the uncorrected, auto-corrected and/or manually corrected R-peaks idx (if present)
    - `bad_segments` contains idx pairs indicating the onsets and offsets of ecg signal segments marked as "bad" using Systole's Editor (optional, only if `manual_correct` is set to True)
    - `info` contains metadata about R-peak detection procedure (e.g., method chosen, artifact correction); its structure changes depending on whether 'sleepecg' or 'neurokit' method was applied
- **6c. (Optional) Export TSV.GZ file with interpolated HR (in bpm)**: a custom function `save_hr_interpol()` saves the `_hr-bpm-{correction_type}.tsv.gz` file with interpolated heart rate (HR) values in bpm from the rr interval time series with selected correction type (i.e., uncorr, autocorr or manualcorr). 

In [ ]:
############## 6. Data output ##############

############# 6a. (Optional) Create and export TSV.GZ file with raw and clean ECG  ##############

# Define function to save TSV.GZ file containing raw and cleaned ECG data
def save_ecg_cleaned(ecg_raw, ecg_clean):
    """ Save raw and cleaned ECG data to a TSV.GZ file.

    Parameters:
    - ecg_raw (ndarray): Array containing raw ECG data (i.e., same as BIDS-compatible _physio.tsv.gz)
    - ecg_clean (ndarray): Array containing cleaned ECG data after 

    Raises:
    - ValueError: If ecg_clean is None.
    """

    # Merge ecg_raw and ecg_clean into one dataframe
    ecg_all = pd.DataFrame({'ecg_raw': ecg_raw, 'ecg_cleaned': ecg_clean})

    # Save as TSV.GZ in derivatives/ecg-preproc/sub-XX/beh/ folder 
    ecg_all_fname = f'{subj_id}_task-{exp_name}_ecg-cleaned.tsv.gz'
    ecg_all_fpath = os.path.join(wd, 'derivatives', 'ecg-preproc', 
                                 subj_id, datatype_name, ecg_all_fname)
    ecg_all.to_csv(ecg_all_fpath, index=False, compression='gzip',
                    sep='\t', na_rep="n/a")

In [ ]:
############# (Optional) Save TSV.GZ file with raw and clean ECG  ##############

# If the optional ECG filtering step was performed, save TSV.GZ file
# (might take a few seconds)
if ecg_filter:
    save_ecg_cleaned(ecg_raw=ecg_arr, ecg_clean=ecg_clean)

In [ ]:
############# 6b. Export main ECG preprocessing info to JSON file  ##############

# Function to convert values to int or None
def convert_to_int_or_none(value):
    if isinstance(value, (float, int, np.float64, np.int64)):
        return int(value) if not np.isnan(value) else None
    return None

# Create a function for data export of main ECG preprocessing info
def save_ecg_preproc(rpeaks_dict, qrs_idx, manual_correct=manual_correct):
    """ Save ECG preprocessing information to a JSON file. 

    Parameters:
    - rpeaks_dict (dict): dictionary containing R-peak detection info from detect_rpeaks().
    - qrs_idx (dict): dictionary containing QRS delineation info from nk.ecg_delineate(). 
    - manual_correct (bool): whether manual R-peak correction was performed with Systole Editor. Default is True.

    Returns:
    - dict: dictionary containing all relevant ECG preprocessing information:
        - 'rpeaks': lists of idx for 'ECG_R_Peaks_Uncorrected', 'ECG_R_Peaks_AutoCorr' (if present) and/or
            'ECG_R_Peaks_ManualCorr' (if present)
        - 'qrs': lists of idx for P, Q, R, S, T events onsets, peaks and/or offsets
        - 'rr_sec': lists of rr interval time series (in seconds) based on uncorrected, auto-corrected and/or
            manually corrected rpeaks (if present)
        - 'bad_segments': list of idx pairs indicating onsets & offsets of bad segments (if present)
        - 'info': metadata about R-peak detection procedure (e.g., method chosen, artifact correction);
            its structure changes depending on whether 'sleepecg' or 'neurokit' method was applied

    Raises:
    - ValueError: If manual_correct is set to True but no corresponding JSON file was found.
    """

    # Intialize dict for saving main ECG preprocessing info
    ecg_preproc_dict = {'rpeaks': {}, 
                        'qrs': {}, 
                        'rr_s': {},
                        'bad_segments': {},
                        'info': {}}
    
    ###### Extract info from automated R-peak detection (and correction) ######

    # Save R-peak idx based on whether automated correction was performed
    if 'ECG_R_Peaks_Uncorrected' in rpeaks_dict['info']:
        rpeaks_uncorr = rpeaks_dict['info']['ECG_R_Peaks_Uncorrected'].tolist()
        rpeaks_auto_corr = rpeaks_dict['info']['ECG_R_Peaks'].tolist()
        rpeaks_auto_corr = [int(x) for x in rpeaks_auto_corr] # convert to int to be JSON-compatible
    else:
        rpeaks_uncorr = rpeaks_dict['info']['ECG_R_Peaks'].tolist()

    rpeaks_uncorr = [int(x) for x in rpeaks_uncorr] # convert to int to be JSON-compatible

    ###### Extract info from Systole's manual R-peak correction ######

    syst_fname = f'{subj_id}_task-{exp_name}_systole-corrected.json'
    syst_fpath = os.path.join(wd, 'derivatives', 'ecg-preproc', 
                                subj_id, datatype_name, syst_fname)

    # Check if manual correction was performed
    if manual_correct and not os.path.exists(syst_fpath):
        raise ValueError(f"Manual correction was set to True, but no corresponding JSON file was found for {subj_id}")

    # Proceed with manual correction if specified
    if manual_correct:
        with open(syst_fpath, 'r') as fsyst:
            syst_rpeaks = json.load(fsyst)

        # Extract values nested under "ecg" and "corrected_peaks"
        rpeaks_manual_corr = syst_rpeaks.get('ecg', {}).get('corrected_peaks', [])
        bad_segments_idx = syst_rpeaks.get('ecg', {}).get('bad_segments', [])

    ###### Extract info from QRS delineation ######

    # Convert qrs_idx columns to int or None for nans to be JSON-compatible
    for key, values in qrs_idx.items():
        qrs_idx[key] = [convert_to_int_or_none(x) for x in values]

    ###### Create rr interval time series (in seconds) ######

    # Convert all available rpeaks idx into rr interval time series 
    rr_s_uncorr = input_conversion(rpeaks_uncorr, input_type="peaks_idx", output_type="rr_s", sfreq=sfreq)
    if 'rpeaks_auto_corr' in locals():
        rr_s_auto_corr = input_conversion(rpeaks_auto_corr, input_type="peaks_idx", output_type="rr_s", sfreq=sfreq)
    if manual_correct: 
        rr_s_manual_corr = input_conversion(rpeaks_manual_corr, input_type="peaks_idx", output_type="rr_s", sfreq=sfreq)

    ###### Store all relevant info from R-peak detection, correction & QRS delineation ######
    
    # Store info regarding R-peak detection and correction
    ecg_preproc_dict['rpeaks']['ECG_R_Peaks_Uncorr'] = rpeaks_uncorr
    if 'rpeaks_auto_corr' in locals():
        ecg_preproc_dict['rpeaks']['ECG_R_Peaks_AutoCorr'] = rpeaks_auto_corr
    if manual_correct: 
        ecg_preproc_dict['rpeaks']['ECG_R_Peaks_ManualCorr'] = rpeaks_manual_corr

    # Store info regarding QRS delineation
    ecg_preproc_dict['qrs'] = qrs_idx

    # Store info regarding rr interval time series (in seconds)
    ecg_preproc_dict['rr_s']['RR_s_Uncorr'] = rr_s_uncorr.tolist()
    if 'rr_s_auto_corr' in locals():
        ecg_preproc_dict['rr_s']['RR_s_AutoCorr'] = rr_s_auto_corr.tolist()
    if manual_correct: 
        ecg_preproc_dict['rr_s']['RR_s_ManualCorr'] = rr_s_manual_corr.tolist()

    # Store info regarding bad segments
    if manual_correct:
        ecg_preproc_dict['bad_segments'] = bad_segments_idx

    # Extract metadata from rpeaks_dict 'info' key
    info_dict = rpeaks_dict['info']
    new_info_dict = {k: v for k, v in info_dict.items() 
                     if k not in ['ECG_R_Peaks', 'ECG_R_Peaks_Uncorrected']} # exclude duplicate info about rpeaks idx
    new_info_dict = {k: (v.tolist() if isinstance(v, np.ndarray) else v) 
                     for k, v in new_info_dict.items()} # convert values to JSON-serializable lists

    # Store metadata regarding rpeak detection procedure
    ecg_preproc_dict['info'] = new_info_dict

    ###### Save JSON file with main info from ECG preprocessing ######

    ecg_preproc_json_fname = f'{subj_id}_task-{exp_name}_ecg-preproc.json'
    ecg_preproc_json_fpath = os.path.join(wd, 'derivatives', 'ecg-preproc', 
                                subj_id, datatype_name, ecg_preproc_json_fname)
    
    with open(ecg_preproc_json_fpath, 'w') as ecg_preproc_json:
        json.dump(ecg_preproc_dict, ecg_preproc_json, allow_nan=True, indent=4)
    
    print(f"ECG preprocessing results have been saved to {ecg_preproc_json_fpath}")

    return ecg_preproc_dict

In [ ]:
############# Export ECG preprocessing JSON with selected options ##############
ecg_preproc_dict = save_ecg_preproc(rpeaks_dict=rpeaks_dict, qrs_idx=qrs_idx)

# ecg_preproc_dict      # uncomment to show

In [ ]:
############# 6c. (Optional) Export TSV.GZ file with interpolated HR (in bpm) ##############

def save_hr_interpol(ecg_preproc_dict, sfreq, rr_type='manualcorr', interpol_type='cubic'): 
    """
    Save interpolated heart rate (HR) values to a TSV file. 

    This function processes RR interval time series from the specified rpeaks correction type, 
    interpolates HR values using Systole utils.heart_rate() function (with cubic interpolation as default) 
    and saves the results to a compressed TSV.GZ file in the derivatives > ecg-preproc folder. 
    
    It requires the existence of the 'ecg_preproc_dict' containing RR intervals, 
    thus the prior section of ECG preprocessing data export (#6b) has to be run beforehand. 

    Parameters:
    - ecg_preproc_dict (dict): dictionary containing info about ecg preprocessing, generated by save_ecg_preproc() (see above). 
    - sfreq (float): sampling frequency of the RR intervals.
    - rr_type (str, optional): type of RR interval correction to use. Must be one of 'uncorr', 'autocorr', or 'manualcorr' (default). 
    - interpol_type (str, optional): type of interpolation to apply. Must be one of 'cubic' (default), 'linear', 'previous', or 'next'.

    Raises:
    - KeyError: if the specified `rr_type` does not exist in `ecg_preproc_dict['rr_s']`.

    Returns:
    - pd.DataFrame: df containing the interpolated HR values.

    Example:
    --------
    >>> save_hr_interpol(ecg_preproc_dict=ecg_preproc_dict, sfreq=1000, rr_type='manualcorr', interpol_type='cubic')
    """

    # Define dict for rr interval correction types
    rr_keys_map = {'uncorr': 'RR_s_Uncorr',
                   'autocorr': 'RR_s_AutoCorr',
                   'manualcorr': 'RR_s_ManualCorr'}
    
    # Check if rr_type is valid
    if rr_type not in rr_keys_map:
        raise ValueError("Invalid rr interval correction type. Please select either 'uncorr', 'autocorr', or 'manualcorr' for rr_type.")
    
    # Check if the corresponding rr correction type key exists in ecg_preproc_dict
    rr_key = rr_keys_map[rr_type]
    if rr_key not in ecg_preproc_dict['rr_s']:
        raise KeyError(f"The rr_type '{rr_type}' cannot be found in ecg_preproc_dict.")
    
    # Save list of rr intervals (in s) for specified correction type
    rr_tmp = ecg_preproc_dict['rr_s'][rr_key]

    # Check if interpol_type is valid
    if interpol_type not in ['cubic', 'linear', 'previous', 'next']:
        raise ValueError("Invalid interpolation type. Please select either 'cubic' (default), 'linear', 'previous', or 'next'.")

    # Interpolate HR values using Systole heart_rate()
    hr_bpm, hr_time = heart_rate(x=rr_tmp, sfreq=sfreq, unit='bpm', kind=interpol_type, input_type='rr_s')

    # Define directory for TSV file with interpolated HR values
    hr_tsv_fname = f'{subj_id}_task-{exp_name}_hr-bpm-{rr_type}.tsv.gz'
    hr_tsv_fpath = os.path.join(wd, 'derivatives', 'ecg-preproc', 
                                subj_id, datatype_name, hr_tsv_fname)
    os.makedirs(os.path.dirname(hr_tsv_fpath), exist_ok=True)

    # Save TSV.GZ file with interpolated HR values (in bpm)
    header = f"hr_bpm_{rr_type}" # header of HR column based on correction type
    hr_df = pd.DataFrame(hr_bpm, columns=[header])
    hr_df.to_csv(hr_tsv_fpath, index=False, compression='gzip', sep='\t', na_rep="n/a")

    print(f"HR interpolation results have been saved to {hr_tsv_fpath}")
    
    return hr_df


In [ ]:
############# (Optional) Export interpolated HR TSV.GZ file with selected options ##############

# If the optional data export of HR interpolation was selected, save TSV.GZ file
# (might take a few seconds)
if hr_interpol: 
    hr_bpm_df = save_hr_interpol(ecg_preproc_dict=ecg_preproc_dict, sfreq=sfreq, interpol_type='cubic')

## **Final note**

The current script was a preliminary development version of the ECG Preprocessing pipeline v0.0.1 from the Brain-Body Analysis Special Interest Group (BBSIG), which was launched on April 2025. For the most recent version, we recommend to check out our repository: https://martager.github.io/bbsig/code/preprocessing/ecg_preproc.ipynb

When using or adapting this BBSIG pipeline to conduct ECG preprocessing in your research work, please cite us in your publication as follows: 

**APA**

*Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://doi.org/10.5281/zenodo.15212797*